# 07.5 — Freeze / unfreeze curriculum

The third curriculum lever decides *which subnetworks learn* each epoch — freeze the classifier while the encoder learns to reconstruct, freeze the encoder while the classifier catches up, then unfreeze everything for joint fine-tuning. Here's the twist, and it's the payoff of this whole module's "read the code, not the label" theme: **freezing is not done with `requires_grad`**. It's done by *scaling each subnetwork's learning rate* — because MATLAB's `setLearnRateFactor` lets a module be "1% frozen," and `requires_grad` is only ever fully on or fully off. This notebook is that mechanism, and why it can't be `requires_grad`.

This notebook covers:

* Per-module optimizer parameter groups (one group per subnetwork).
* Freezing = writing `group["lr"] = base_lr × factor`, not `requires_grad = False`.
* Why the `1e-2` "slow-learn" factor makes `requires_grad` impossible.
* The freeze curriculum shape and how it dovetails with the loss weights.

**Prerequisites:** [05.4 LR scheduling](../05_training_loop/05.4_learning_rate_scheduling.ipynb), [07.4 loss weights curriculum](07.4_loss_weights_curriculum.ipynb).

## Section 1 — What MATLAB does

`cgg_setFrozenNetwork_v2` reads a per-subnetwork `CurrentFactor*` and applies it with `setLearnRateFactor`:

```matlab
% For each subnetwork, set a LEARN-RATE FACTOR (a multiplier), not a boolean:
factor_enc = cgg_calculateDynamicValue(FreezeParameters.Encoder, epoch);   % e.g. 1e-2
net = setLearnRateFactor(net, 'encoder', factor_enc);   % scales encoder's LR

% factor 1.0 = full learning, 0.0 = frozen, 1e-2 = "mostly frozen, learning slowly"
```

`setLearnRateFactor` multiplies each learnable's effective learning rate by the factor. Crucially, MATLAB's freeze schedule uses `1e-2` — *not* `0` — for the "frozen" stages, so a module keeps inching along (momentum stays alive) rather than stopping dead. That fractional factor is the whole reason the port can't use a boolean `requires_grad`.

## Section 2 — The Python concepts you need

### 2.1 — Per-module optimizer parameter groups

PyTorch optimizers accept *parameter groups* — subsets of parameters, each with its own hyperparameters (like `lr`). The port builds **one group per named subnetwork** (encoder, decoder, classifier), so each can have an independent learning rate:

In [ ]:
import torch
import torch.nn as nn
from neural_data_decoding.training.freezing import build_optimizer_with_module_groups

torch.manual_seed(0)
modules = {"encoder": nn.Linear(4, 4), "decoder": nn.Linear(4, 4), "classifier": nn.Linear(4, 2)}
opt = build_optimizer_with_module_groups(modules, initial_lr=0.1)

for g in opt.param_groups:
    print(f"group '{g['name']}': {len(g['params'])} tensors, lr={g['lr']}")
print("→ one param group per subnetwork, each with its OWN lr — the handle freezing turns.")

### 2.2 — Freezing = scaling the group's learning rate

In [ ]:
from neural_data_decoding.training.schedules.factory import make_freeze_schedule, ScheduleWaypoints
from neural_data_decoding.training.freezing import apply_freeze_to_optimizer

# Freeze schedule: classifier mostly-frozen (1e-2) until epoch 10, unfreeze by 15.
freeze = make_freeze_schedule(encoder=1.0, decoder=1.0, classifier=1.0,
    waypoints={"classifier": ScheduleWaypoints.of((0, 10, 15), (1e-2, 1e-2, 1.0))})

for epoch in (1, 12, 15):
    freeze.update(epoch)
    apply_freeze_to_optimizer(opt, freeze, base_lr=0.1)
    lrs = {g["name"]: round(g["lr"], 4) for g in opt.param_groups}
    print(f"epoch {epoch:2}: lrs = {lrs}")
print("→ apply_freeze writes group['lr'] = base_lr × factor. classifier 'frozen' = lr 0.001, not off.")

`apply_freeze_to_optimizer` reads each subnetwork's current freeze *factor* and writes `group["lr"] = base_lr × factor`. A factor of `1e-2` gives the classifier `lr = 0.001` — it *is* learning, just 100× slower than the unfrozen encoder. No `requires_grad` toggling anywhere.

### 2.3 — Factor 0 freezes; factor 1e-2 slow-learns; factor 1 is full

In [ ]:
def move_after_steps(factor, steps=5):
    torch.manual_seed(0)
    m = nn.Linear(3, 3)
    o = build_optimizer_with_module_groups({"m": m}, initial_lr=0.1)
    for g in o.param_groups:
        g["lr"] = 0.1 * factor                 # apply the freeze factor
    w0 = m.weight.detach().clone()
    for _ in range(steps):
        o.zero_grad(); (m(torch.randn(2, 3)) ** 2).mean().backward(); o.step()
    return (m.weight.detach() - w0).abs().sum().item()

for factor in (0.0, 1e-2, 1.0):
    print(f"factor {factor:>4}: weights moved by {move_after_steps(factor):.4f}")
print("→ 0.0 frozen (no movement), 1e-2 slow-learn (a little), 1.0 full. A CONTINUOUS dial.")

### 2.4 — Why not `requires_grad`? Because `1e-2` exists

Here's the crux. `requires_grad = False` is a *boolean*: a parameter either trains or it doesn't. But MATLAB's freeze schedule uses `1e-2` — "mostly frozen, but still learning slowly to keep momentum alive." There is **no way** to express "learn at 1% speed" with `requires_grad`; it can only do 0% or 100%. To reproduce MATLAB's `setLearnRateFactor` semantics, the port *must* scale the learning rate, which is a continuous dial (§2.3). This is why the README's "requires_grad management" description is wrong — and why reading the actual code matters more than the label (the recurring lesson of [06.7](../06_loss_orchestration/06.7_the_confidence_pd_controller.ipynb), [07.2](07.2_piecewise_linear_schedules.ipynb)).

In [ ]:
# requires_grad is binary — it cannot express the 1e-2 slow-learn factor:
p = nn.Parameter(torch.randn(3))
print("requires_grad can be:", True, "or", False, "— that's it. No '1% frozen'.")
print("The freeze factors the curriculum actually uses:", [1.0, 1e-2, 0.0], "← 1e-2 needs a CONTINUOUS knob.")
print("→ only per-group LR scaling can represent a fractional freeze. requires_grad can't.")

There's a secondary reason too: with `AdamW`, a group at `lr = 0` still *runs* the optimizer step — the adaptive moments keep updating — but the weight update is scaled to zero, so nothing moves. When the module later unfreezes, its moment estimates are already warm ([02.7](../02_numpy_and_pytorch_basics/02.7_optimizers.ipynb)). Toggling `requires_grad` would instead yank the parameter in and out of the graph, discarding that continuity.

### 2.5 — The freeze curriculum, and the dovetail with loss weights

In [ ]:
import matplotlib.pyplot as plt
from neural_data_decoding.training.schedules.interpolator import piecewise_anneal_value

# Soft Three-Stage freeze: encoder/decoder unfrozen→mostly-frozen→unfrozen; classifier frozen→unfrozen.
enc = [piecewise_anneal_value(1.0, [20, 25, 45, 46], [1.0, 1e-2, 1e-2, 1.0], e) for e in range(1, 50)]
cls = [piecewise_anneal_value(1.0, [0, 10, 15], [1e-2, 1e-2, 1.0], e) for e in range(1, 50)]

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(range(1, 50), enc, "o-", label="encoder / decoder LR factor", color="#1a4a7a", markersize=3)
ax.plot(range(1, 50), cls, "d-", label="classifier LR factor", color="crimson", markersize=3)
ax.set_yscale("log"); ax.set_xlabel("epoch"); ax.set_ylabel("freeze factor (log scale)")
ax.set_title("Freeze curriculum: train AE first, then classifier, then joint fine-tune")
ax.legend()
plt.tight_layout(); plt.show()
print("Classifier unfreezes as the encoder is learning; encoder mostly-freezes while the classifier catches up.")

Read the two curves together with the loss weights ([07.4](07.4_loss_weights_curriculum.ipynb)):

* **Epochs 1–15:** encoder/decoder full (`1.0`), classifier mostly-frozen (`1e-2`) then unfreezing — the AE learns to reconstruct while the classifier waits, matching the reconstruct-first loss weights.
* **Epochs 25–45:** encoder/decoder *mostly-frozen* (`1e-2`) while the classifier (now full) learns on a stable latent — freeze the representation so the classifier can't corrupt it.
* **Epoch 46+:** encoder/decoder unfreeze again for joint fine-tuning.

The freeze lever and the weight lever are two views of the *same* curriculum: don't just down-weight a loss, also stop the subnetwork it trains. Together they enact the staged handoff ([05.6](../05_training_loop/05.6_the_two_stage_lifecycle.ipynb)) smoothly.

## Section 3 — The `neural_data_decoding` implementation

`apply_freeze_to_optimizer` — the LR-factor write, and note the *absence* of `requires_grad`:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/training/freezing.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if "for group in optimizer.param_groups:" in line)
for k in range(i, i + 6):
    print(f"{k + 1:4} | {src[k]}")
# Confirm there is no requires_grad anywhere in the freezing module:
has_rg = any("requires_grad" in line for line in src)
print(f"\n'requires_grad' appears in freezing.py? {has_rg}  ← freezing is purely LR-scaling.")

Things to spot:

* The function writes `group["lr"] = base_lr * freeze_schedule.current(name)` for each named group — that's the entire freeze mechanism. Search the file for `requires_grad`: it isn't there.
* It mirrors `cgg_setFrozenNetwork_v2`'s `setLearnRateFactor` — a *multiplier* per subnetwork, which only per-group LR scaling can reproduce (§2.4).
* `build_optimizer_with_module_groups` de-duplicates shared parameters by `id()` — each learnable lands in exactly one group ("one learnable, one freeze factor"), so a parameter can't get two conflicting factors.
* Groups whose names aren't in the freeze schedule keep their previous `lr` — only scheduled subnetworks are touched.
* A frozen group (`lr = 0` or `1e-2`) still steps through `AdamW`; the update just scales down (§2.4) — verified by `test_apply_freeze_does_not_step_frozen_params`.

## Section 4 — Hands-on exercises

### Exercise 4.1 — the frozen classifier still has warm moments

Freeze a module (factor 0), run some steps, then unfreeze (factor 1) and show it moves *immediately* — because AdamW kept its moments warm while frozen.

In [ ]:
# Reveal:
torch.manual_seed(0)
m = nn.Linear(3, 3)
o = build_optimizer_with_module_groups({"m": m}, initial_lr=0.1)
data = [torch.randn(2, 3) for _ in range(3)]

for g in o.param_groups: g["lr"] = 0.0            # frozen
w_before = m.weight.detach().clone()
for x in data:
    o.zero_grad(); (m(x) ** 2).mean().backward(); o.step()   # moments update, weights don't
print("while frozen (lr=0), weight moved:", round((m.weight.detach() - w_before).abs().sum().item(), 5))

for g in o.param_groups: g["lr"] = 0.1            # unfreeze
w_before = m.weight.detach().clone()
o.zero_grad(); (m(data[0]) ** 2).mean().backward(); o.step()
print("first step after unfreezing, weight moved:", round((m.weight.detach() - w_before).abs().sum().item(), 5))
print("→ the frozen period kept moments warm; unfreezing resumes learning instantly.")

### Exercise 4.2 — `requires_grad` can't do what the curriculum needs

Enumerate the freeze factors the Soft Three-Stage regime uses and show that a boolean can only represent two of the three.

In [ ]:
# Reveal:
factors_used = {1.0: "unfrozen", 1e-2: "mostly-frozen (slow-learn)", 0.0: "fully frozen"}
print("freeze factors in the curriculum:")
for f, desc in factors_used.items():
    representable = "requires_grad=True" if f == 1.0 else ("requires_grad=False" if f == 0.0 else "NOT REPRESENTABLE")
    print(f"  {f:>5}: {desc:28} → {representable}")
print("→ the 1e-2 slow-learn factor has no requires_grad equivalent. LR scaling is mandatory.")

## Section 5 — Common errors

### A "frozen" module is still learning (a little)

Expected if the factor is `1e-2` — that's slow-learn, not off (§2.2). Use factor `0.0` for a hard freeze. MATLAB deliberately uses `1e-2` to keep momentum alive.

### Using `requires_grad = False` to freeze

It works for a *hard* freeze, but it can't express the `1e-2` slow-learn factor (§2.4), and it changes graph participation (dropping AdamW's warm moments, §2.4). Match the reference: scale the group LR.

### The freeze schedule has no effect

`apply_freeze_to_optimizer` must be called each epoch, *and* the optimizer must have named per-module groups (§2.1). A single-group optimizer has no per-subnetwork handle to scale — `build_optimizer_with_module_groups` is required.

### A shared parameter gets the wrong factor

`build_optimizer_with_module_groups` de-duplicates by `id()` so each parameter is in one group. If a parameter is shared across "encoder" and "decoder," it lands in whichever group is built first — check the intended freeze factor applies.

### Freeze and loss-weight schedules disagree

They should be consistent (§2.5): don't down-weight a loss while leaving its subnetwork fully trainable (or vice versa). The Soft Three-Stage regime aligns them — the freeze and weight curricula are designed together.

## Section 6 — Further reading

- [`src/neural_data_decoding/training/freezing.py`](../../src/neural_data_decoding/training/freezing.py) — `build_optimizer_with_module_groups`, `apply_freeze_to_optimizer`.
- [05.4 learning rate scheduling](../05_training_loop/05.4_learning_rate_scheduling.ipynb) — the `param_group["lr"]` mechanism freezing reuses.
- [`torch.optim` parameter groups](https://pytorch.org/docs/stable/optim.html#per-parameter-options) — the PyTorch feature underneath.

Next up: **[07.6 walkthrough — the soft three-stage curriculum](07.6_walkthrough_soft_three_stage_curriculum_shortened.ipynb)** — all three levers traced together, end to end.